# ViTASA Enhanced — Pair Classification (Colab — Git version)

Pull from GitHub → Install → Train → Download results

**Time: ~2 hours (4 configs × 3 domains × 10 epochs)**


In [ ]:
# 1. Clone your GitHub repo
!git clone https://github.com/YOUR_GITHUB/VITASA_Enhanced.git 2>&1 | tail -5
!cd VITASA_Enhanced && pwd && ls -la | head -20

In [ ]:
# 2. Install dependencies
!pip install -q torch transformers scikit-learn seqeval underthesea
import torch
print(f"✅ PyTorch {torch.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (slow!)'}")

In [ ]:
# 3. Smoke test (1 epoch, 10% data) — 5 phút
%cd VITASA_Enhanced
!python3 train_pair.py --domain mobile --loss ce --epochs 1 --subsample 0.1 --batch-size 64 2>&1 | tail -20
print("\n✅ Smoke test passed!")

In [ ]:
# 4. RUN ALL — 4 configs × 3 domains × 10 epochs (~2 hours)
import subprocess
import time

EPOCHS = 10
BATCH_SIZE = 64
CONFIGS = [
    ("C1", "--loss ce"),
    ("C2", "--loss ce --normalize"),
    ("C3", "--loss focal"),
    ("C4", "--loss focal --normalize"),
]
DOMAINS = ['mobile', 'restaurant', 'hotel']

print(f"Starting {len(CONFIGS)*len(DOMAINS)} runs (~{len(CONFIGS)*len(DOMAINS)*EPOCHS//5} min total)\n")

for domain in DOMAINS:
    for config_name, flags in CONFIGS:
        print(f"[{domain}/{config_name}] {EPOCHS} epochs...")
        cmd = f"python3 train_pair.py --domain {domain} {flags} --epochs {EPOCHS} --batch-size {BATCH_SIZE}"
        result = subprocess.run(cmd.split())
        if result.returncode != 0:
            print(f"ERROR: {domain}/{config_name} failed!")
            break
        time.sleep(3)

print("\n✅ All done!")

In [ ]:
# 5. Print summary
import json
from pathlib import Path
from collections import defaultdict

results_dir = Path("experiments/results_pair")
results = defaultdict(dict)
BASELINE = {"mobile": 61.77, "restaurant": 41.12, "hotel": 52.64}

for f in sorted(results_dir.glob("*/results.json")):
    try:
        d = json.load(open(f))
        results[d["domain"]][d["config"]] = d["test"]["macro_f1"] * 100
    except: pass

print("\n" + "="*90)
print("ABLATION RESULTS — macro F1 (3 sentiment classes)")
print("="*90)
print(f"{'Domain':<12} {'C1_baseline':>15} {'C2_norm':>15} {'C3_focal':>15} {'C4_full':>15} {'Baseline':>13}")
print("-" * 90)

for domain in ['mobile', 'restaurant', 'hotel']:
    c1, c2, c3, c4 = None, None, None, None
    for k, v in results.get(domain, {}).items():
        if 'loss-ce_visobert' in k and 'norm' not in k: c1 = v
        elif 'loss-ce_norm' in k: c2 = v
        elif 'loss-focal_visobert' in k and 'norm' not in k: c3 = v
        elif 'loss-focal_norm' in k: c4 = v
    
    print(f"{domain:<12} {(f'{c1:.2f}%' if c1 else '—'):>15} {(f'{c2:.2f}%' if c2 else '—'):>15} {(f'{c3:.2f}%' if c3 else '—'):>15} {(f'{c4:.2f}%' if c4 else '—'):>15} {BASELINE[domain]:>12.2f}%")

print("="*90)

In [ ]:
# 6. Download results
from google.colab import files
!tar -czf results.tar.gz experiments/results_pair/
files.download("results.tar.gz")
print("✅ Downloaded!")